# Block 2 — Frozen clinical knowledge graph

Catalogue, profile expansion, tube rules, `assume()`, `validate_request`.

Live Block 3 does **not** call `assume()` — that is Block 4. This notebook uses `KnowledgeGraph.load()` only. No PHI.


## 0. Clone the live tree


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/RwaRwa599/epq3.git"
BRANCH = "block1"


def _run(cmd):
    print("$", " ".join(str(c) for c in cmd))
    subprocess.check_call(cmd)


def ensure_med_doc() -> Path:
    """Clone live tree if needed and put src/ on sys.path (Colab pip -e is unreliable)."""
    here = Path.cwd().resolve()
    candidates = [
        here,
        here.parent,
        Path("/content/epq3"),
        Path("/content") / "epq3",
    ]
    root = None
    for cand in candidates:
        if (cand / "src" / "med_doc" / "__init__.py").is_file() and (cand / "pyproject.toml").is_file():
            root = cand
            break
    if root is None:
        dest = Path("/content/epq3") if Path("/content").is_dir() else (here / "epq3")
        url = REPO
        token = os.environ.get("GITHUB_TOKEN")
        if not token:
            try:
                from google.colab import userdata
                token = userdata.get("GITHUB_TOKEN")
            except Exception:
                token = None
        if token:
            url = f"https://{token}@github.com/RwaRwa599/epq3.git"
        if not (dest / ".git").is_dir():
            _run(["git", "clone", "--branch", BRANCH, "--single-branch", url, str(dest)])
        else:
            _run(["git", "-C", str(dest), "fetch", "origin", BRANCH])
            _run(["git", "-C", str(dest), "checkout", BRANCH])
            _run(["git", "-C", str(dest), "pull", "--ff-only", "origin", BRANCH])
        root = dest
    os.chdir(root)
    src = str((root / "src").resolve())
    if src not in sys.path:
        sys.path.insert(0, src)
    os.environ["PYTHONPATH"] = src + os.pathsep + os.environ.get("PYTHONPATH", "")
    try:
        _run([sys.executable, "-m", "pip", "install", "-q", "matplotlib", "opencv-python-headless", "pydantic", "Pillow", "numpy"])
        _run([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    except Exception as exc:
        print("pip note (sys.path still has src/):", exc)
    import med_doc
    print("cwd:", os.getcwd())
    print("med_doc:", med_doc.__file__)
    return root

root = ensure_med_doc()


In [ ]:
import os
import sys
from pathlib import Path

def _ensure_src_on_path():
    for cand in [Path.cwd(), Path("/content/epq3"), Path.cwd().parent]:
        src = cand / "src"
        if (src / "med_doc" / "__init__.py").is_file():
            os.chdir(cand)
            sp = str(src.resolve())
            if sp not in sys.path:
                sys.path.insert(0, sp)
            return sp
    dest = Path("/content/epq3") if Path("/content").is_dir() else (Path.cwd() / "epq3")
    if not (dest / "src" / "med_doc" / "__init__.py").is_file():
        import subprocess
        subprocess.check_call(
            [
                "git",
                "clone",
                "--branch",
                "block1",
                "--single-branch",
                "https://github.com/RwaRwa599/epq3.git",
                str(dest),
            ]
        )
    os.chdir(dest)
    sp = str((dest / "src").resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)
    return sp

_ensure_src_on_path()

from pathlib import Path
import json
import cv2
import matplotlib.pyplot as plt

try:
    from google.colab import files as colab_files
except Exception:
    colab_files = None

OUT = Path("/content/pipeline") if Path("/content").is_dir() else Path("outputs/colab_pipeline")
OUT.mkdir(parents=True, exist_ok=True)

def show_rgb(path, title="", figsize=(10, 8)):
    path = Path(path)
    if not path.exists():
        print("missing", path)
        return
    bgr = cv2.imread(str(path))
    if bgr is None:
        print("unreadable", path)
        return
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=figsize)
    plt.imshow(rgb)
    plt.title(title or path.name)
    plt.axis("off")
    plt.show()

def download(path):
    path = Path(path)
    print(path, f"({path.stat().st_size / 1024:.1f} KB)" if path.exists() else "missing")
    if colab_files and path.exists():
        colab_files.download(str(path))

def demo_sheet() -> Path:
    from med_doc.paths import SYNTHETIC_DIR
    sheet = SYNTHETIC_DIR / "lab_request_v0_blank.png"
    assert sheet.exists(), sheet
    return sheet

## 1. Load `kg/lab_request_v1_kg.json`


In [ ]:
import os
import sys
from pathlib import Path

def _ensure_src_on_path():
    for cand in [Path.cwd(), Path("/content/epq3"), Path.cwd().parent]:
        src = cand / "src"
        if (src / "med_doc" / "__init__.py").is_file():
            os.chdir(cand)
            sp = str(src.resolve())
            if sp not in sys.path:
                sys.path.insert(0, sp)
            return sp
    dest = Path("/content/epq3") if Path("/content").is_dir() else (Path.cwd() / "epq3")
    if not (dest / "src" / "med_doc" / "__init__.py").is_file():
        import subprocess
        subprocess.check_call(
            [
                "git",
                "clone",
                "--branch",
                "block1",
                "--single-branch",
                "https://github.com/RwaRwa599/epq3.git",
                str(dest),
            ]
        )
    os.chdir(dest)
    sp = str((dest / "src").resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)
    return sp

_ensure_src_on_path()

from med_doc.kg import KnowledgeGraph
from med_doc.paths import DEFAULT_KG

kg = KnowledgeGraph.load()
print("kg file:", DEFAULT_KG)
print("version:", kg.version)
print("catalogue items:", len(kg.catalogue))
print("profiles:", sorted(kg.profile_bundles)[:12], "...")

## 2. Profiles, tubes, aliases


In [ ]:
print("lipid bundle:", kg.expand_profile("profile_lipid"))
print("cbc tubes:", kg.calculate_expected_tubes(["cbc"]))
print("cbc+glucose:", kg.calculate_expected_tubes(["cbc", "glucose_fasting"]))
print("resolve SGPT:", kg.resolve_alias("SGPT"))
ranked = kg.assume("others", "triglyc", {"ticked_ids": ["profile_lipid"]}, top_k=3)
for c in ranked:
    print(f"  assume {c.canonical_id!r:20} tier={c.tier} score={c.score:.3f} {c.reason}")

## 3. `validate_request` (observed tubes, not a copy of expected)


In [ ]:
ok = kg.validate_request(["cbc"], observed_tubes={"EDTA": 1})
short = kg.validate_request(["cbc"], observed_tubes={"EDTA": 0})
missing = kg.validate_request(["cbc"], observed_tubes={})
print("match   valid", ok.is_valid, "expected", ok.expected_tubes)
print("short   valid", short.is_valid, short.discrepancies)
print("missing valid", missing.is_valid, "expected", missing.expected_tubes, "(empty obs is not auto-filled)")